# Limpieza de matriculados
Se consolidan las tres materias en `data/matriculados/matriculados.parquet`. Se omite el título del Excel y se excluyen las filas de conteo y las filas vacías. Solo se completa `instancia` con `Libre` cuando está vacía y `estado_insc` es `Baja`.

También se normalizan encabezados y espacios, se convierten fechas e identificadores numéricos y se agrega la materia y el archivo de origen. Por cada alumno (usuario) y materia se conserva únicamente el registro con el mayor número de transacción, cualquiera sea su estado. Los controles detienen la exportación si quedan instancias nulas o datos obligatorios inválidos.

In [1]:
from pathlib import Path
import pandas as pd
import sys

# Permite ejecutar desde la raíz del proyecto o desde jupyter/.
raiz = Path.cwd() if (Path.cwd() / 'data/matriculados').is_dir() else Path.cwd().parent

sys.path.insert(0, str(raiz / "jupyter"))
from utils_limpieza import estandarizar_usuario

carpeta = raiz / 'data/matriculados'
archivos = sorted(a for a in carpeta.glob('*.xlsx') if not a.name.startswith('~$'))
if len(archivos) != 3:
    raise ValueError(f'Se esperaban 3 archivos de materias; se encontraron {len(archivos)}.')

def a_snake_case(columnas):
    return (columnas.str.strip().str.lower()
            .str.normalize('NFKD').str.encode('ascii', 'ignore').str.decode('ascii')
            .str.replace(r'\W+', '_', regex=True).str.strip('_'))


In [2]:
dfs = []
resumen = []
columnas = ['alumno', 'identificacion', 'comision', 'estado_insc', 'instancia',
            'fecha_inscripcion', 'nro_de_transaccion']

for archivo in archivos:
    df = pd.read_excel(archivo, header=1)
    df.columns = a_snake_case(df.columns)
    df = df.dropna(axis=1, how='all')
    if set(df.columns) != set(columnas):
        raise ValueError(f'{archivo.name}: encabezados inesperados: {df.columns.tolist()}')
    df = df[columnas].copy()

    # Los blancos y las cadenas de espacios se consideran nulos.
    for columna in df.select_dtypes(include=['object', 'string']).columns:
        df[columna] = df[columna].map(lambda valor: valor.strip() if isinstance(valor, str) else valor)
        df[columna] = df[columna].replace(r'^\s*$', pd.NA, regex=True)

    es_conteo = df['alumno'].astype('string').str.fullmatch(
        r'(?i)cantidad\s+de\s+filas\s*:?\s*', na=False)
    es_vacia = df.isna().all(axis=1)
    df = df.loc[~(es_conteo | es_vacia)].copy()

    completar = df['instancia'].isna() & df['estado_insc'].astype('string').str.casefold().eq('baja').fillna(False)
    df.loc[completar, 'instancia'] = 'Libre'

    # errors='raise' evita ocultar valores inválidos mediante conversiones a nulo.
    df['fecha_inscripcion'] = pd.to_datetime(df['fecha_inscripcion'], format='%d/%m/%Y %H:%M:%S', errors='raise')
    for columna in ['comision', 'nro_de_transaccion']:
        df[columna] = pd.to_numeric(df[columna], errors='raise').astype('Int64')
    for columna in ['alumno', 'identificacion', 'estado_insc', 'instancia']:
        df[columna] = df[columna].astype('string')

    # Clave de estudiante con formato ALU_xxxx
    df['usuario'] = estandarizar_usuario(df['identificacion'], archivo.name)

    nulos = df[columnas].isna().sum()
    if nulos.any():
        raise ValueError(f'{archivo.name}: quedan nulos que requieren revisión: {nulos[nulos > 0].to_dict()}')

    df['materia'] = archivo.stem
    df['archivo'] = archivo.name
    resumen.append({
        'materia': archivo.stem, 'filas': len(df),
        'filas_conteo_excluidas': int(es_conteo.sum()),
        'filas_vacias_excluidas': int(es_vacia.sum()),
        'instancias_completadas': int(completar.sum()),
        'filas_duplicadas': int(df.duplicated().sum()),
        'identificaciones_repetidas': int(df['identificacion'].duplicated().sum()),
        'transacciones_repetidas': int(df['nro_de_transaccion'].duplicated().sum()),
    })
    dfs.append(df)

# Se agrega 'usuario' al orden de columnas
df_matriculados = pd.concat(dfs, ignore_index=True)[['archivo', 'materia', 'usuario', *columnas]]
# La última inscripción se determina por transacción, dentro de cada alumno y materia.
filas_antes = len(df_matriculados)
df_matriculados = (df_matriculados
                   .sort_values('nro_de_transaccion', kind='stable')
                   .drop_duplicates(subset=['materia', 'usuario'], keep='last')  # 'usuario' en vez de 'identificacion'
                   .sort_index().reset_index(drop=True))
print(f'Inscripciones anteriores excluidas: {filas_antes - len(df_matriculados)}')
resumen_limpieza = pd.DataFrame(resumen)
resumen_limpieza['filas_exportadas'] = resumen_limpieza['materia'].map(df_matriculados.groupby('materia').size())
display(resumen_limpieza)
display(df_matriculados.isna().sum().rename('nulos'))

Inscripciones anteriores excluidas: 14


,materia,filas,filas_conteo_excluidas,filas_vacias_excluidas,instancias_completadas,filas_duplicadas,identificaciones_repetidas,transacciones_repetidas,filas_exportadas
0,Algoritmos y Estructuras de Datos,175,1,0,10,0,8,0,167
1,Probabilidad y Estadística,171,1,0,7,0,4,0,167
2,Álgebra,167,1,0,4,0,2,0,165


archivo               0
materia               0
usuario               0
alumno                0
identificacion        0
comision              0
estado_insc           0
instancia             0
fecha_inscripcion     0
nro_de_transaccion    0
Name: nulos, dtype: int64

In [4]:
salida = carpeta / 'matriculados.parquet'
df_matriculados.to_parquet(salida, index=False)
# Verifica que la lectura conserve filas, valores y tipos.
pd.testing.assert_frame_equal(df_matriculados, pd.read_parquet(salida))
print(f'Exportadas {len(df_matriculados)} inscripciones a {salida}')
display(df_matriculados.head())


Exportadas 499 inscripciones a c:\Users\DELL\Documents\2026-2C-BIA BORRAR EN 2027\TPG FINAL\py.tpf.business.analytics\data\matriculados\matriculados.parquet


,archivo,materia,usuario,alumno,identificacion,comision,estado_insc,instancia,fecha_inscripcion,nro_de_transaccion
0,Algoritmos y Estructuras de Datos.xlsx,Algoritmos y Estructuras de Datos,ALU_1248,"PÉREZ, FELIPE",ALU_1248,2320,Aceptada,Regularidad,2025-07-31 11:06:04,298521
1,Algoritmos y Estructuras de Datos.xlsx,Algoritmos y Estructuras de Datos,ALU_1212,"ORTIZ, GONZALO",ALU_1212,2320,Aceptada,Regularidad,2025-08-04 21:13:11,299608
2,Algoritmos y Estructuras de Datos.xlsx,Algoritmos y Estructuras de Datos,ALU_1089,"MORENO, BRUNO",ALU_1089,2320,Aceptada,Regularidad,2025-08-04 00:43:56,299233
3,Algoritmos y Estructuras de Datos.xlsx,Algoritmos y Estructuras de Datos,ALU_1016,"FERRER, JULIETA",ALU_1016,2320,Aceptada,Regularidad,2025-09-24 18:23:34,303668
4,Algoritmos y Estructuras de Datos.xlsx,Algoritmos y Estructuras de Datos,ALU_1186,"RUIZ, MARCOS",ALU_1186,2320,Aceptada,Regularidad,2025-08-18 12:24:53,301039
